# 04 - Tự viết pass và dựng benchmark harness

Ba notebook trước dùng pass có sẵn. Notebook này làm hai việc để chuẩn bị cho nghiên cứu:

1. **Tự viết pass** - `AnalysisPass` và `TransformationPass`, rồi gắn vào pipeline. Một RL agent
   muốn chạy trong Qiskit thật thì cuối cùng phải đóng gói thành một `TransformationPass`.
2. **Dựng benchmark harness** - module `bench.py` cạnh notebook này, dùng lại ở notebook `05` và ở
   folder `03_`.

Việc thứ hai nghe nhàm nhưng quan trọng hơn. Nhìn lại ba notebook trước thì thấy số liệu routing
dao động rất mạnh: ±9% do seed, ±16% do layout, và khác nhau 3 lần do topology. Trong một môi trường
nhiễu như thế, **kết luận "agent của tôi tốt hơn" rất dễ sai**, và sai theo hướng mình muốn tin.
Harness tồn tại để chặn đúng chuyện đó.

In [ ]:
import numpy as np
import pandas as pd

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.random import random_circuit
from qiskit.quantum_info import Operator
from qiskit.transpiler import (
    AnalysisPass,
    CouplingMap,
    PassManager,
    TransformationPass,
    generate_preset_pass_manager,
)
from qiskit.transpiler.passes import CheckMap, SabreSwap

import bench

print("bench.py nằm cạnh notebook, các hàm chính:")
print(" ", [t for t in dir(bench) if not t.startswith("_") and callable(getattr(bench, t))])

## 1. `AnalysisPass`: đọc DAG, ghi `property_set`

Pass phân tích **không** được sửa DAG. Nó đọc, tính toán, rồi ghi kết quả vào
`self.property_set` - một dict dùng chung cho cả pipeline, và là cách duy nhất các pass nói chuyện
với nhau. `run()` trả về `None`.

In [ ]:
class ThongKeMach(AnalysisPass):
    """Thu các metric của mạch vào property_set["thong_ke"]."""

    def __init__(self, nhan=""):
        super().__init__()
        self.nhan = nhan

    def run(self, dag):
        ops = dag.count_ops()
        tk = {
            "nhan": self.nhan,
            "depth": dag.depth(),
            "size": dag.size(),
            "hai_qubit": len(dag.two_qubit_ops()),
            "swap": ops.get("swap", 0),
        }
        # nối thêm vào danh sách để xem được tiến triển qua nhiều điểm trong pipeline
        self.property_set["thong_ke"] = (self.property_set["thong_ke"] or []) + [tk]
        return None          # AnalysisPass KHÔNG trả về dag


qc = QuantumCircuit(3)
qc.h(0)
qc.swap(0, 1)
qc.swap(0, 1)          # hai SWAP liên tiếp cùng cặp -> triệt tiêu
qc.cx(1, 2)
qc.swap(1, 2)
qc.swap(2, 1)          # viết ngược thứ tự qubit, vẫn triệt tiêu
qc.t(0)

pm = PassManager([ThongKeMach("trước")])
pm.run(qc)
print("thống kê:", pm.property_set["thong_ke"])

## 2. `TransformationPass`: trả về DAG mới

Pass biến đổi nhận `dag`, sửa nó (hoặc dựng cái mới) và **trả về** DAG. Dưới đây là một pass thật:
huỷ hai SWAP liên tiếp trên cùng cặp qubit. Đây là tối ưu đúng đắn vì
$\mathrm{SWAP} \cdot \mathrm{SWAP} = I$.

Hai chi tiết dễ sai:

- Phải so sánh **tập** qubit (`set(node.qargs)`), vì `swap(1,2)` và `swap(2,1)` là cùng một cổng.
- Hai SWAP phải **kề nhau trên DAG** (cổng sau là successor trực tiếp của cổng trước trên *cả hai*
  qubit), không chỉ gần nhau trong danh sách lệnh - nếu có cổng khác chen giữa thì không triệt tiêu
  được.

In [ ]:
class HuySwapLienKe(TransformationPass):
    """Huỷ cặp SWAP liên tiếp trên cùng cặp qubit (SWAP . SWAP = I)."""

    def run(self, dag):
        con_tim = True
        while con_tim:
            con_tim = False
            for node in dag.op_nodes():
                if node.name != "swap":
                    continue
                for ke in dag.successors(node):
                    if getattr(ke, "name", None) != "swap":
                        continue
                    if set(ke.qargs) != set(node.qargs):
                        continue
                    # kề nhau trên cả hai qubit: không có cổng nào chen giữa
                    dag.remove_op_node(node)
                    dag.remove_op_node(ke)
                    con_tim = True
                    break
                if con_tim:
                    break
        return dag


pm = PassManager([ThongKeMach("trước"), HuySwapLienKe(), ThongKeMach("sau")])
ket_qua = pm.run(qc)

print(pd.DataFrame(pm.property_set["thong_ke"]).to_string(index=False))
print("\nmạch vẫn đúng:", Operator(ket_qua).equiv(Operator(qc)))
print("-> 4 SWAP biến thành 0. Kiểm tra bằng Operator là bắt buộc:")
print("   một pass 'tối ưu' mà làm sai mạch thì luôn thắng mọi benchmark.")

### Gắn pass vào pipeline thật

Hai cách:

**(a) `PassManager` riêng** - như các ô trên, dùng khi muốn kiểm soát hoàn toàn.

**(b) Thay một stage của preset pass manager** - gán vào `pm.routing`, `pm.layout`, ... Cách này giữ
nguyên toàn bộ phần còn lại của pipeline chuẩn (translation, optimization, scheduling) và chỉ đổi
phần mình quan tâm. Đây chính là cách một RL router sẽ được tích hợp.

In [ ]:
CM = CouplingMap.from_line(5)
mach_thu = bench.mach_ngau_nhien(5, 10, seed=11)

print(f"{'routing stage':24s} {'cx':>5s} {'depth':>6s} {'hợp lệ':>7s}")
for ten, stage in [
    ("mặc định (SabreSwap)", None),
    ("thay bằng basic", PassManager([SabreSwap(CM, heuristic="basic", seed=0)])),
    ("thay bằng basic + huỷ swap", PassManager([SabreSwap(CM, heuristic="basic", seed=0),
                                                HuySwapLienKe()])),
]:
    pm = generate_preset_pass_manager(optimization_level=2, basis_gates=bench.BASIS,
                                      coupling_map=CM, seed_transpiler=1)
    if stage is not None:
        pm.routing = stage
    out = pm.run(mach_thu)
    chk = PassManager([CheckMap(CM)])
    chk.run(out)
    print(f"{ten:24s} {out.count_ops().get('cx', 0):5d} {out.depth():6d} "
          f"{str(chk.property_set['is_swap_mapped']):>7s}")

print("\n-> Lưu ý: đếm 'swap' sau cả pipeline sẽ luôn ra 0, vì stage translation")
print("   đã dịch mỗi SWAP thành 3 cx. Muốn đếm SWAP thì phải đếm NGAY SAU routing.")

## 3. Benchmark harness

Bốn nguyên tắc của `bench.py`, mỗi cái chặn một kiểu kết luận sai:

| Nguyên tắc | Chặn lỗi gì |
|---|---|
| luôn chạy `CheckMap`, raise nếu không hợp lệ | agent "thắng" bằng cách bỏ qua cổng khó |
| **tách** seed của mạch khỏi seed của router | lẫn "mạch này khó hơn" với "router dao động" |
| cố định layout khi so router | so sai bài toán (layout đáng ~16%, hơn khác biệt giữa router) |
| đếm SWAP ngay sau routing | sau translation thì `count_ops()["swap"]` luôn là 0 |

Nguyên tắc thứ hai là cái tôi làm sai ở lần dựng đầu: dùng chung một `seed` cho cả mạch và router,
nên `std` trong bảng vô nghĩa - nó trộn hai nguồn biến thiên khác bản chất.

In [ ]:
df = bench.chay_benchmark(n_qubit=9, do_sau=20, seeds_mach=range(3), seeds_router=range(4))

print(f"{len(df)} lần đo, tất cả hợp lệ: {df['hop_le'].all()}, "
      f"tổng thời gian {df['giay'].sum():.2f}s")
print(f"mạch: 9 qubit, {df['cx_goc'].min()}-{df['cx_goc'].max()} cổng cx\n")
print("số SWAP trung bình (mean trên mọi seed):")
print(bench.bang_tom_tat(df, "swap").to_string())
print("\ndepth trung bình:")
print(bench.bang_tom_tat(df, "depth").to_string())

Hai bảng trên nói hai chuyện khác nhau, và đó là điểm cần chú ý khi chọn reward: `grid(3,3)` ít SWAP
hơn `ring(9)`, nhưng xem cột depth thì thứ hạng có thể đổi. Tối ưu một thước đo không tự động tối ưu
thước đo kia.

### Nhiễu do seed - con số cần biết trước khi tuyên bố cải thiện

In [ ]:
print("Dao động khi GIỮ CỐ ĐỊNH mạch và topology, chỉ đổi seed của router:")
print(bench.nhieu_router(df, "swap").to_string())
print()
print("-> BasicSwap là deterministic: std = 0, không có seed nào ảnh hưởng.")
print("-> SabreSwap(decay) dao động ~9% chỉ vì seed.")
print("   Nghĩa là: một agent báo 'tốt hơn SabreSwap 5%' mà chỉ chạy một seed")
print("   thì con số đó NẰM TRONG nhiễu, không chứng minh được gì.")

### So sánh ghép cặp, không so hai số trung bình

Độ khó giữa các mạch chênh nhau rất nhiều (bảng trên: 25 tới 112 SWAP). Nếu lấy trung bình của
router A trên một tập mạch rồi so với trung bình của router B trên tập mạch khác, kết quả phản ánh
tập mạch nhiều hơn phản ánh router.

`so_voi_baseline` ghép từng cặp theo đúng `(topology, mạch, seed)` rồi mới lấy trung bình của hiệu.

In [ ]:
print("So với SabreSwap(decay), ghép theo từng (topology, mạch, seed):")
print(bench.so_voi_baseline(df, baseline="SabreSwap(decay)", chi_so="swap").to_string(index=False))
print()
print("-> BasicSwap tệ hơn 82%, SabreSwap(basic) tệ hơn 10%.")
print("-> Đây là khoảng cách mà một RL agent có thể nhắm vào: nếu agent chỉ bằng")
print("   SabreSwap(basic) thì nó vẫn còn kém mặc định của Qiskit 10%.")

## 4. Minh hoạ cái bẫy: một cấu hình có thể "chứng minh" kết luận ngược

Ở trên, tổng hợp 48 cặp cho thấy `decay` hơn `basic` 10%. Nhưng trong 48 cặp đó có vài cặp mà
`basic` **thắng**. Nếu báo cáo đúng một cặp như vậy - và chọn nó sau khi đã xem kết quả - thì "chứng
minh" được điều ngược lại.

Đây không phải tình huống giả định: đó chính là hình dạng của một bảng số liệu RL chỉ chạy một seed
và một mạch.

In [ ]:
p = df.pivot_table(index=["topology", "seed_mach", "seed_router"], columns="router", values="swap")
basic, decay = p["SabreSwap(basic)"], p["SabreSwap(decay)"]

nguoc = p[basic < decay]
print(f"Trong {len(p)} cặp đo: decay thắng {(decay < basic).sum()}, "
      f"hoà {(decay == basic).sum()}, THUA {(basic < decay).sum()}\n")
print("Các cặp mà basic thắng - chọn một dòng là 'chứng minh' được basic tốt hơn:")
print(nguoc.to_string())

# chọn đúng cặp có lợi nhất cho basic - chính là việc một người muốn tin sẽ làm
hang = nguoc.loc[(nguoc["SabreSwap(basic)"] / nguoc["SabreSwap(decay)"]).idxmin()]
print(f"\nVí dụ, chỉ báo cáo cặp có lợi nhất: basic {hang['SabreSwap(basic)']:.0f} swap "
      f"vs decay {hang['SabreSwap(decay)']:.0f} swap")
print(f"-> 'basic tốt hơn {(1 - hang['SabreSwap(basic)'] / hang['SabreSwap(decay)']) * 100:.0f}%'")
print("   Đúng số liệu, sai kết luận. Cách chặn: cố định tập mạch và tập seed TRƯỚC khi đo.")

### Dùng harness với router của riêng mình

`chay_benchmark` nhận tham số `routers` là một callable `(cm, seed) -> dict`. Nên gắn `mini_router`
từ notebook `02`, hay một RL policy ở folder `03_`, chỉ cần bọc nó thành một `TransformationPass`
rồi trả về trong dict đó - mọi bảng số liệu ở trên lập tức dùng được.

In [ ]:
class RouterNgauNhien(TransformationPass):
    """Router chọn SWAP ngẫu nhiên - baseline DƯỚI cùng cho mọi so sánh RL."""

    def __init__(self, coupling_map, seed=0):
        super().__init__()
        self.cm = coupling_map
        self.D = coupling_map.distance_matrix
        self.rng = np.random.default_rng(seed)
        self.edges = sorted(set(tuple(sorted(e)) for e in coupling_map.get_edges()))

    def run(self, dag):
        from qiskit.converters import circuit_to_dag
        from qiskit.dagcircuit import DAGCircuit

        pi = list(range(dag.num_qubits()))
        moi = DAGCircuit()
        for qreg in dag.qregs.values():
            moi.add_qreg(qreg)
        for creg in dag.cregs.values():
            moi.add_creg(creg)
        qubits = list(dag.qubits)

        while dag.op_nodes():
            tien_do = False
            for node in list(dag.front_layer()):
                qs = [dag.find_bit(q).index for q in node.qargs]
                if len(qs) == 1 or self.D[pi[qs[0]]][pi[qs[1]]] == 1:
                    moi.apply_operation_back(node.op, [qubits[pi[q]] for q in qs], node.cargs)
                    dag.remove_op_node(node)
                    tien_do = True
            if tien_do:
                continue
            cap = [[dag.find_bit(q).index for q in nd.qargs]
                   for nd in dag.front_layer() if len(nd.qargs) == 2]
            ket = {pi[q] for c in cap for q in c}
            ung_vien = [e for e in self.edges if e[0] in ket or e[1] in ket]
            a, b = ung_vien[self.rng.integers(len(ung_vien))]
            la, lb = pi.index(a), pi.index(b)
            pi[la], pi[lb] = pi[lb], pi[la]
            from qiskit.circuit.library import SwapGate
            moi.apply_operation_back(SwapGate(), [qubits[a], qubits[b]], ())
        return moi


def bo_router_mo_rong(cm, seed=0):
    bo = bench.bo_router(cm, seed)
    bo["RandomPolicy"] = RouterNgauNhien(cm, seed=seed)
    return bo


df2 = bench.chay_benchmark(n_qubit=5, do_sau=12, seeds_mach=range(3), seeds_router=range(3),
                           routers=bo_router_mo_rong)
print(f"{len(df2)} lần đo, hợp lệ: {df2['hop_le'].all()}\n")
print(bench.so_voi_baseline(df2, baseline="SabreSwap(decay)").to_string(index=False))
print("\n-> Random policy là mốc DƯỚI. Một RL agent phải vượt nó rất xa mới đáng nói,")
print("   và phải tiến tới SabreSwap(decay) mới có ý nghĩa thực tế.")

## Tóm tắt

| Nội dung | Chi tiết |
|---|---|
| `AnalysisPass` | `run(dag)` → ghi `self.property_set`, trả về `None` |
| `TransformationPass` | `run(dag)` → trả về DAG mới |
| `property_set` | kênh duy nhất để các pass nói chuyện với nhau |
| dựng DAG mới | `DAGCircuit()` + `add_qreg` + `apply_operation_back` |
| gắn vào pipeline | `PassManager([...])`, hoặc gán `pm.routing = ...` của preset |
| đếm SWAP | **ngay sau routing** - sau translation thì luôn là 0 |
| kiểm tra pass | `Operator(sau).equiv(Operator(trước))` - pass sai luôn thắng benchmark |
| nhiễu do seed | `SabreSwap(decay)` ~9%; `BasicSwap` deterministic |
| so sánh | **ghép cặp** theo (topology, mạch, seed), không so hai số trung bình |
| mốc dưới / mốc trên | `RandomPolicy` / `SabreSwap(decay)` |
| khoảng để cải thiện | basic kém decay 10%, BasicSwap kém 82% |

`bench.py` xuất ra: `mach_ngau_nhien`, `bo_topology`, `bo_router`, `do_mot_lan`,
`chay_benchmark`, `bang_tom_tat`, `nhieu_router`, `xep_hang`, `so_voi_baseline`.

### Thử tự làm

1. Viết một `AnalysisPass` tính **xác suất thành công** của mạch (hàm `xac_suat_khong_loi` ở
   notebook `01`) rồi ghi vào `property_set`. Dùng nó làm `chi_so` cho `bang_tom_tat`.
2. `HuySwapLienKe` hiện chỉ huỷ SWAP **kề nhau**. Mở rộng: nếu giữa hai SWAP chỉ có cổng 1-qubit
   trên các qubit khác thì vẫn huỷ được. Nó tìm thêm được bao nhiêu cặp trên output của
   `SabreSwap`?
3. Chạy `chay_benchmark` với `n_qubit=16` và `bo_topology(16)` (có `grid(4,4)`). Thời gian chạy
   tăng thế nào? Thứ hạng router có đổi không?
4. Thêm `gom_lookahead=True` vào `bo_router` và chạy với `n_qubit=5`. `LookaheadSwap` có đáng cái
   giá thời gian của nó ở kích thước này không?
5. Với cùng tập mạch, so `TrivialLayout` và `VF2Layout` (truyền qua `layout` của `do_mot_lan`).
   Bao nhiêu phần trăm cải thiện đến từ layout, bao nhiêu từ router?